## After Job1
### Before Job2


In [0]:
%pip install faiss-cpu sentence-transformers

In [0]:
# The purpose of code is to validate: befor joining IMDB data, there's 42% missing in description and all fields, and after we join IMDB, what do we have ?

# ── Tier distribution validator ───────────────────────────────────────────────
# Paste into a new cell in Job1_Embeddings_Final_260427 or any new notebook.
# Reads the same files Job1 produced. No recompute — read-only.


import pandas as pd
import os

OUTPUTS_DIR       = "/Volumes/movie_recsys/data/outputs"
META_CLEAN_PATH   = f"{OUTPUTS_DIR}/meta_clean.parquet"
TMDB_CHECKPOINT   = f"{OUTPUTS_DIR}/tmdb_enriched.parquet"
MOST_HELPFUL_PATH = f"{OUTPUTS_DIR}/most_helpful.parquet"

import sys
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')
from features import build_embedding_input, get_embedding_tier

# ── 1. Load all three sources ─────────────────────────────────────────────────
meta        = pd.read_parquet(META_CLEAN_PATH)
most_helpful = pd.read_parquet(MOST_HELPFUL_PATH)
tmdb        = pd.read_parquet(TMDB_CHECKPOINT)

print(f"meta rows       : {len(meta):,}")
print(f"most_helpful rows: {len(most_helpful):,}")
print(f"tmdb rows        : {len(tmdb):,}")

# ── 2. Merge exactly as Job1 does ─────────────────────────────────────────────
meta = meta.merge(most_helpful, on="parent_asin", how="left")
meta = meta.merge(tmdb,         on="parent_asin", how="left")

# ── 3. Coalesce: Amazon first, TMDB fallback ──────────────────────────────────
def _is_present(v):
    if v is None: return False
    if isinstance(v, float) and pd.isna(v): return False
    s = str(v).strip()
    return bool(s) and s.lower() != "nan"

def _clean(v):
    return str(v).strip() if _is_present(v) else None

def _coalesce(primary, fallback):
    return _clean(primary) if _is_present(primary) else _clean(fallback)

meta["title_final"]       = meta.apply(lambda r: _coalesce(r["title"],          r.get("tmdb_title")),       axis=1)
meta["genres_final"]      = meta.apply(lambda r: _coalesce(r["genres_str"],      r.get("tmdb_genres")),      axis=1)
meta["description_final"] = meta.apply(lambda r: _coalesce(r["description_str"], r.get("tmdb_description")), axis=1)

# ── 4. Assign tiers ───────────────────────────────────────────────────────────
meta["embedding_tier"] = meta.apply(
    lambda r: get_embedding_tier(
        r["title_final"], r["genres_final"],
        r["description_final"], r.get("most_helpful_review")
    ), axis=1
)

# ── 5. Print distribution ─────────────────────────────────────────────────────
total = len(meta)
dist  = meta["embedding_tier"].value_counts().sort_index()

print("\n── Tier distribution (post-TMDB enrichment) ──────────────────")
for tier, count in dist.items():
    bar = "█" * int(count / total * 40)
    print(f"  Tier {tier}: {count:>7,}  ({100*count/total:5.1f}%)  {bar}")

print(f"\n  TOTAL    : {total:>7,}  (100.0%)")
print(f"\n  Embeddable (Tier 1–3): {total - dist.get(4,0):>7,}  ({100*(total-dist.get(4,0))/total:.1f}%)")
print(f"  Excluded   (Tier 4)  : {dist.get(4,0):>7,}  ({100*dist.get(4,0)/total:.1f}%)")

# ── 6. Field-level coverage breakdown ─────────────────────────────────────────
print("\n── Field coverage after coalesce ─────────────────────────────")
for col, label in [
    ("title_final",       "title      "),
    ("genres_final",      "genres     "),
    ("description_final", "description"),
    ("most_helpful_review","review     "),
]:
    n_filled = meta[col].notna().sum() if col in meta.columns else 0
    print(f"  {label}: {n_filled:>7,}  ({100*n_filled/total:.1f}%)")

# ── 7. Tier 4 pre vs post TMDB comparison ────────────────────────────────────
# Pre-TMDB tier (Amazon-only fields, before coalesce)
meta["tier_pre_tmdb"] = meta.apply(
    lambda r: get_embedding_tier(
        _clean(r.get("title")), _clean(r.get("genres_str")),
        _clean(r.get("description_str")), r.get("most_helpful_review")
    ), axis=1
)

pre_t4  = (meta["tier_pre_tmdb"] == 4).sum()
post_t4 = dist.get(4, 0)
rescued = pre_t4 - post_t4

print("\n── TMDB rescue impact ────────────────────────────────────────")
print(f"  Tier 4 before TMDB : {pre_t4:>7,}  ({100*pre_t4/total:.1f}%)")
print(f"  Tier 4 after  TMDB : {post_t4:>7,}  ({100*post_t4/total:.1f}%)")
print(f"  Items rescued      : {rescued:>7,}  ({100*rescued/total:.2f}%)")

In [0]:
# Layer 1 — Mechanical checks (did it finish?)
# These are binary. Run them in a new Databricks cell immediately after the notebook completes.

import os
import numpy as np
import faiss

OUTPUTS = "/Volumes/movie_recsys/data/outputs"

# 1. Files exist
assert os.path.exists(f"{OUTPUTS}/faiss_index.bin"), "FAIL: faiss_index.bin missing"
assert os.path.exists(f"{OUTPUTS}/embeddings.npy"),  "FAIL: embeddings.npy missing"
assert os.path.exists(f"{OUTPUTS}/tmdb_enriched.parquet"), "FAIL: TMDB checkpoint missing"

# 2. Embeddings shape is correct
embeddings = np.load(f"{OUTPUTS}/embeddings.npy")
assert embeddings.ndim == 2,        f"FAIL: expected 2D array, got {embeddings.ndim}D"
assert embeddings.shape[1] == 384,  f"FAIL: expected 384 dims, got {embeddings.shape[1]}"
assert embeddings.shape[0] > 190000, f"FAIL: only {embeddings.shape[0]} items embedded, expected ~200K"
print(f"✓ Embeddings: {embeddings.shape[0]:,} items × {embeddings.shape[1]} dims")

# 3. FAISS index is consistent with embeddings
index = faiss.read_index(f"{OUTPUTS}/faiss_index.bin")
assert index.ntotal == embeddings.shape[0], \
    f"FAIL: index has {index.ntotal} vectors but embeddings has {embeddings.shape[0]}"
assert index.d == 384, f"FAIL: index dim {index.d} != 384"
print(f"✓ FAISS index: {index.ntotal:,} vectors, dim={index.d}")

print("\n✓ Layer 1 passed — all mechanical checks clean")

In [0]:
# Layer 2 — Quantitative checks (did it work correctly?)
# These catch silent failures — embeddings that generated but are degenerate (all zeros, all identical, NaN).

# 4. Embeddings are not degenerate
norms = np.linalg.norm(embeddings, axis=1)
assert norms.min() > 0.01,    f"FAIL: near-zero embeddings found (min norm={norms.min():.4f})"
assert not np.isnan(norms).any(), "FAIL: NaN values in embeddings"

# Check embeddings are not all identical (would mean model failed silently)
sample_idx = np.random.choice(len(embeddings), 1000, replace=False)
sample = embeddings[sample_idx]
pairwise_std = sample.std(axis=0).mean()
assert pairwise_std > 0.01, f"FAIL: embeddings have near-zero variance ({pairwise_std:.4f}) — model may have failed"
print(f"✓ Embedding variance healthy (mean std across dims: {pairwise_std:.4f})")

# 5. Tier distribution — confirm TMDB bridge ran
tier_report_path = f"{OUTPUTS}/tier_distribution.json"
# Or read it from the notebook's printed output and verify:
# Tier 1 (full):   ~26% of items
# Tier 4 (bridge): ~42% of items → should be near-zero AFTER TMDB enrichment
# If Tier 4 is still ~42% after the job, TMDB loop did not run

# 6. FAISS search returns results (not crashing)
query = embeddings[0:1]  # use first item as a query
distances, indices = index.search(query, 10)
assert indices.shape == (1, 10), "FAIL: FAISS search returned wrong shape"
assert (indices >= 0).all(),     "FAIL: FAISS returned -1 indices (untrained index?)"
assert distances[0][0] < 0.01,  "FAIL: nearest neighbour of item to itself should be ~0"
print(f"✓ FAISS search working — top-10 distances: {distances[0].round(3)}")

print("\n✓ Layer 2 passed — quantitative checks clean")

In [0]:
# Layer 3 — Semantic spot-check (did it learn anything meaningful?)
# This is the one you do with your eyes. It's in the blueprint explicitly — "CB (embeddings) — qualitative spot-check: Pick 10 seed movies. Show top 5 FAISS nearest neighbours for each. Verify semantic coherence."

from sentence_transformers import SentenceTransformer
import pandas as pd

# Load title lookup
meta = spark.read.parquet(f"{OUTPUTS}/meta_clean.parquet") \
            .select("asin", "title").toPandas()
asin_to_title = dict(zip(meta["asin"], meta["title"]))

# Load the asin ordering used during embedding
# (Job 1 notebook should have saved this — if not, this is a gap to fix)
asin_order = pd.read_parquet(f"{OUTPUTS}/asin_order.parquet")["asin"].tolist()

# Spot-check function
model = SentenceTransformer("all-MiniLM-L6-v2")

def spot_check(seed_title_fragment, k=5):
    # Find the asin whose title contains the fragment
    match = meta[meta["title"].str.contains(seed_title_fragment, case=False, na=False)]
    if match.empty:
        print(f"'{seed_title_fragment}' not found in metadata"); return
    seed_asin = match.iloc[0]["asin"]
    seed_idx  = asin_order.index(seed_asin)
    seed_emb  = embeddings[seed_idx:seed_idx+1]
    distances, indices = index.search(seed_emb, k+1)  # +1 because result[0] is itself
    print(f"\nSeed: {asin_to_title.get(seed_asin, seed_asin)}")
    print(f"{'─'*60}")
    for rank, (dist, idx) in enumerate(zip(distances[0][1:], indices[0][1:]), 1):
        title = asin_to_title.get(asin_order[idx], asin_order[idx])
        print(f"  {rank}. {title}  (dist={dist:.3f})")

# The three examples the blueprint asks for in the README
spot_check("Martian")       # should find Interstellar, Gravity, Apollo 13
spot_check("Shawshank")     # should find other prison dramas, not comedies
spot_check("Dora the Explorer")  # should stay in children's content

In [0]:
import numpy as np
emb = np.load("/Volumes/movie_recsys/data/outputs/embeddings.npy")
asin = np.load("/Volumes/movie_recsys/data/outputs/asin_index.npy", 
               allow_pickle=True)
print(f"embeddings : {emb.shape}")
print(f"asin_index : {len(asin)}")

# Check if 256000 is suspicious
import pandas as pd
meta = pd.read_parquet("/Volumes/movie_recsys/data/outputs/meta_clean.parquet")
print(f"meta_clean : {len(meta):,} items")
print(f"Coverage   : {len(emb)/len(meta)*100:.1f}%")

In [0]:
import os
import numpy as np
import sys

OUTPUTS_DIR      = "/Volumes/movie_recsys/data/outputs"
EMBEDDINGS_PATH  = f"{OUTPUTS_DIR}/embeddings.npy"
ASIN_INDEX_PATH  = f"{OUTPUTS_DIR}/asin_index.npy"
TMDB_CHECKPOINT  = f"{OUTPUTS_DIR}/tmdb_enriched.parquet"
MOST_HELPFUL_PATH= f"{OUTPUTS_DIR}/most_helpful.parquet"

import pandas as pd

# How many items survived to the embeddable stage?
meta      = pd.read_parquet(f"{OUTPUTS_DIR}/meta_clean.parquet")
tmdb      = pd.read_parquet(TMDB_CHECKPOINT)
most_helpful = pd.read_parquet(MOST_HELPFUL_PATH)

print(f"meta_clean rows        : {len(meta):,}")
print(f"tmdb_enriched rows     : {len(tmdb):,}")
print(f"most_helpful rows      : {len(most_helpful):,}")

# Reconstruct what embeddable would have looked like
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')
from features import build_embedding_input

meta = meta.merge(most_helpful, on="parent_asin", how="left")
meta = meta.merge(tmdb, on="parent_asin", how="left")

def _is_present(v):
    if v is None: return False
    import math
    if isinstance(v, float) and math.isnan(v): return False
    return bool(str(v).strip()) and str(v).strip().lower() != "nan"

def coalesce(a, b):
    return str(a).strip() if _is_present(a) else (str(b).strip() if _is_present(b) else None)

meta["title_final"]       = meta.apply(lambda r: coalesce(r.get("title"), r.get("tmdb_title")), axis=1)
meta["genres_final"]      = meta.apply(lambda r: coalesce(r.get("genres_str"), r.get("tmdb_genres")), axis=1)
meta["description_final"] = meta.apply(lambda r: coalesce(r.get("description_str"), r.get("tmdb_description")), axis=1)
meta["review"]            = meta.get("most_helpful_review", None)

meta["embedding_input"] = meta.apply(
    lambda r: build_embedding_input(
        r["title_final"], r["genres_final"],
        r["description_final"], r.get("most_helpful_review"),
    ), axis=1
)

true_gaps  = meta["embedding_input"].isna()
embeddable = meta[~true_gaps]

print(f"\nembeddazble rows        : {len(embeddable):,}")
print(f"true gaps              : {true_gaps.sum():,}")
print(f"currently embedded     : 256,000")
print(f"missing from index     : {len(embeddable) - 256000:,}")

# Check if 256000 is a batch boundary
print(f"\n256000 / 512 (batch size) = {256000 / 512:.0f} batches exactly")
print(f"This tells us: the loop stopped at exactly batch 500")

In [0]:
# STEP A: Diagnose the current state and check for 500-batch cap

import sys
sys.path.append('/Workspace/Users/mqwebster238@gmail.com/novametrics/src/')
from features import build_embedding_input
import pandas as pd
import numpy as np

OUTPUTS_DIR = "/Volumes/movie_recsys/data/outputs"

# Reconstruct what embeddable SHOULD contain
meta = pd.read_parquet(f"{OUTPUTS_DIR}/meta_clean.parquet")
most_helpful = pd.read_parquet(f"{OUTPUTS_DIR}/most_helpful.parquet")
tmdb = pd.read_parquet(f"{OUTPUTS_DIR}/tmdb_enriched.parquet")

meta = meta.merge(most_helpful, on="parent_asin", how="left")
meta = meta.merge(tmdb, on="parent_asin", how="left")

def _is_present(v):
    if v is None: return False
    import math
    if isinstance(v, float) and math.isnan(v): return False
    return bool(str(v).strip()) and str(v).strip().lower() != "nan"

def coalesce(a, b):
    return str(a).strip() if _is_present(a) else (str(b).strip() if _is_present(b) else None)

meta["title_final"] = meta.apply(lambda r: coalesce(r.get("title"), r.get("tmdb_title")), axis=1)
meta["genres_final"] = meta.apply(lambda r: coalesce(r.get("genres_str"), r.get("tmdb_genres")), axis=1)
meta["description_final"] = meta.apply(lambda r: coalesce(r.get("description_str"), r.get("tmdb_description")), axis=1)

meta["embedding_input"] = meta.apply(
    lambda r: build_embedding_input(
        r["title_final"], r["genres_final"],
        r["description_final"], r.get("most_helpful_review"),
    ), axis=1
)

true_gaps = meta["embedding_input"].isna()
embeddable = meta[~true_gaps]

# Current checkpoint state
emb_current = np.load(f"{OUTPUTS_DIR}/embeddings.npy")
asin_current = np.load(f"{OUTPUTS_DIR}/asin_index.npy", allow_pickle=True)

print("="*70)
print("DIAGNOSIS: Job 1 Embedding Loop Status")
print("="*70)
print(f"\nExpected embeddable items : {len(embeddable):,}")
print(f"Actually embedded items   : {len(emb_current):,}")
print(f"Missing from checkpoint   : {len(embeddable) - len(emb_current):,}")
print(f"\nCheckpoint calculations:")
print(f"  256,000 / 512 = {256000 / 512:.0f} batches (if stopped at batch 500)")
print(f"  433,586 / 512 = {len(embeddable) / 512:.1f} batches (full dataset needs ~848 batches)")
print(f"\nConclusion: Loop stopped at exactly batch 500 out of ~848 needed.")
print(f"\nResume point: batch {len(emb_current) // 512} (item {len(emb_current):,})")
print("="*70)